In [1]:
import torch
import torch.nn as nn

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
df = pd.read_csv('Data/NYCTaxiFares.csv')

In [3]:
df.head()

,pickup_datetime,fare_amount,fare_class,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2010-04-19 08:17:56 UTC,6.5,0,-73.992365,40.730521,-73.975499,40.744746,1
1,2010-04-17 15:43:53 UTC,6.9,0,-73.990078,40.740558,-73.974232,40.744114,1
2,2010-04-17 11:23:26 UTC,10.1,1,-73.994149,40.751118,-73.960064,40.766235,2
3,2010-04-11 21:25:03 UTC,8.9,0,-73.990485,40.756422,-73.971205,40.748192,1
4,2010-04-17 02:19:01 UTC,19.7,1,-73.990976,40.734202,-73.905956,40.743115,1


In [4]:
df['fare_amount'].describe()

count    120000.000000
mean         10.040326
std           7.500134
min           2.500000
25%           5.700000
50%           7.700000
75%          11.300000
max          49.900000
Name: fare_amount, dtype: float64

In [5]:
def haversine_distance(df, lat1, long1, lat2, long2):
    """
    Calculates the haversine distance between 2 sets of GPS coordinates in df
    """
    r = 6371
    phi1 = np.radians(df[lat1])
    phi2 = np.radians(df[lat2])
    
    delta_phi = np.radians(df[lat2]-df[lat1])
    delta_lambda = np.radians(df[long2]-df[long1])
    
    a = np.sin(delta_phi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(delta_lambda/2)**2
    c = 2*np.arctan2(np.sqrt(a), np.sqrt(1-a))
    d = (r*c) # in kilometers
    return d


In [6]:
df['dist_km'] = haversine_distance(df,'pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude')   

In [7]:
df.head()

,pickup_datetime,fare_amount,fare_class,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,dist_km
0,2010-04-19 08:17:56 UTC,6.5,0,-73.992365,40.730521,-73.975499,40.744746,1,2.126312
1,2010-04-17 15:43:53 UTC,6.9,0,-73.990078,40.740558,-73.974232,40.744114,1,1.392307
2,2010-04-17 11:23:26 UTC,10.1,1,-73.994149,40.751118,-73.960064,40.766235,2,3.326763
3,2010-04-11 21:25:03 UTC,8.9,0,-73.990485,40.756422,-73.971205,40.748192,1,1.864129
4,2010-04-17 02:19:01 UTC,19.7,1,-73.990976,40.734202,-73.905956,40.743115,1,7.231321


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   pickup_datetime    120000 non-null  object 
 1   fare_amount        120000 non-null  float64
 2   fare_class         120000 non-null  int64  
 3   pickup_longitude   120000 non-null  float64
 4   pickup_latitude    120000 non-null  float64
 5   dropoff_longitude  120000 non-null  float64
 6   dropoff_latitude   120000 non-null  float64
 7   passenger_count    120000 non-null  int64  
 8   dist_km            120000 non-null  float64
dtypes: float64(6), int64(2), object(1)
memory usage: 8.2+ MB


In [9]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   pickup_datetime    120000 non-null  datetime64[ns, UTC]
 1   fare_amount        120000 non-null  float64            
 2   fare_class         120000 non-null  int64              
 3   pickup_longitude   120000 non-null  float64            
 4   pickup_latitude    120000 non-null  float64            
 5   dropoff_longitude  120000 non-null  float64            
 6   dropoff_latitude   120000 non-null  float64            
 7   passenger_count    120000 non-null  int64              
 8   dist_km            120000 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(6), int64(2)
memory usage: 8.2 MB


In [10]:
my_time = df['pickup_datetime'][0]

In [11]:
my_time.hour

8

In [12]:
df['EDTdate'] = df['pickup_datetime'] - pd.Timedelta(hours=4)
df['Hour'] = df['EDTdate'].dt.hour
df['AMorPM'] = np.where(df['Hour']<12,'am','pm')
df['Weekday'] = df['EDTdate'].dt.strftime("%a")
df.head()


,pickup_datetime,fare_amount,fare_class,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,dist_km,EDTdate,Hour,AMorPM,Weekday
0,2010-04-19 08:17:56+00:00,6.5,0,-73.992365,40.730521,-73.975499,40.744746,1,2.126312,2010-04-19 04:17:56+00:00,4,am,Mon
1,2010-04-17 15:43:53+00:00,6.9,0,-73.990078,40.740558,-73.974232,40.744114,1,1.392307,2010-04-17 11:43:53+00:00,11,am,Sat
2,2010-04-17 11:23:26+00:00,10.1,1,-73.994149,40.751118,-73.960064,40.766235,2,3.326763,2010-04-17 07:23:26+00:00,7,am,Sat
3,2010-04-11 21:25:03+00:00,8.9,0,-73.990485,40.756422,-73.971205,40.748192,1,1.864129,2010-04-11 17:25:03+00:00,17,pm,Sun
4,2010-04-17 02:19:01+00:00,19.7,1,-73.990976,40.734202,-73.905956,40.743115,1,7.231321,2010-04-16 22:19:01+00:00,22,pm,Fri


In [13]:
cat_cols = ['Hour', 'AMorPM', 'Weekday']
cont_cols = ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude', 'passenger_count', 'dist_km']
y_col = ['fare_amount']


In [14]:
df.dtypes

pickup_datetime      datetime64[ns, UTC]
fare_amount                      float64
fare_class                         int64
pickup_longitude                 float64
pickup_latitude                  float64
dropoff_longitude                float64
dropoff_latitude                 float64
passenger_count                    int64
dist_km                          float64
EDTdate              datetime64[ns, UTC]
Hour                               int32
AMorPM                            object
Weekday                           object
dtype: object

In [15]:
for cat in cat_cols:
    df[cat] = df[cat].astype('category')

In [16]:
df.dtypes

pickup_datetime      datetime64[ns, UTC]
fare_amount                      float64
fare_class                         int64
pickup_longitude                 float64
pickup_latitude                  float64
dropoff_longitude                float64
dropoff_latitude                 float64
passenger_count                    int64
dist_km                          float64
EDTdate              datetime64[ns, UTC]
Hour                            category
AMorPM                          category
Weekday                         category
dtype: object

In [17]:
df['Hour'].head()

0     4
1    11
2     7
3    17
4    22
Name: Hour, dtype: category
Categories (24, int32): [0, 1, 2, 3, ..., 20, 21, 22, 23]

In [18]:
df['AMorPM'].head()

0    am
1    am
2    am
3    pm
4    pm
Name: AMorPM, dtype: category
Categories (2, object): ['am', 'pm']

In [19]:
df['Weekday'].head()

0    Mon
1    Sat
2    Sat
3    Sun
4    Fri
Name: Weekday, dtype: category
Categories (7, object): ['Fri', 'Mon', 'Sat', 'Sun', 'Thu', 'Tue', 'Wed']

In [20]:
df["Weekday"].cat.codes # This is how the computer sees the data. It converts the data into numbers

0         1
1         2
2         2
3         3
4         0
         ..
119995    3
119996    0
119997    3
119998    5
119999    2
Length: 120000, dtype: int8

In [21]:
df['Weekday'].cat.codes.values # Now we have the data in a numpy array

array([1, 2, 2, ..., 3, 5, 2], dtype=int8)

In [22]:
hr = df['Hour'].cat.codes.values
ampm = df['AMorPM'].cat.codes.values
wkdy = df['Weekday'].cat.codes.values

In [23]:
hr

array([ 4, 11,  7, ..., 14,  4, 12], dtype=int8)

In [24]:
cats = np.stack([hr, ampm, wkdy], 1)
cats

array([[ 4,  0,  1],
       [11,  0,  2],
       [ 7,  0,  2],
       ...,
       [14,  1,  3],
       [ 4,  0,  5],
       [12,  1,  2]], dtype=int8)

In [25]:
cats = torch.tensor(cats, dtype=torch.int64)
cats

tensor([[ 4,  0,  1],
        [11,  0,  2],
        [ 7,  0,  2],
        ...,
        [14,  1,  3],
        [ 4,  0,  5],
        [12,  1,  2]])

In [26]:
conts = np.stack([df[col].values for col in cont_cols], 1)
conts

array([[ 40.730521  , -73.992365  ,  40.744746  , -73.975499  ,
          1.        ,   2.12631159],
       [ 40.740558  , -73.990078  ,  40.744114  , -73.974232  ,
          1.        ,   1.39230687],
       [ 40.751118  , -73.994149  ,  40.766235  , -73.960064  ,
          2.        ,   3.32676344],
       ...,
       [ 40.749772  , -73.988574  ,  40.707799  , -74.011541  ,
          3.        ,   5.05252282],
       [ 40.724529  , -74.004449  ,  40.730765  , -73.992697  ,
          1.        ,   1.20892296],
       [ 40.77192   , -73.955415  ,  40.763015  , -73.967623  ,
          3.        ,   1.42739869]])

In [27]:
conts = torch.tensor(conts, dtype=torch.float)
conts

tensor([[ 40.7305, -73.9924,  40.7447, -73.9755,   1.0000,   2.1263],
        [ 40.7406, -73.9901,  40.7441, -73.9742,   1.0000,   1.3923],
        [ 40.7511, -73.9941,  40.7662, -73.9601,   2.0000,   3.3268],
        ...,
        [ 40.7498, -73.9886,  40.7078, -74.0115,   3.0000,   5.0525],
        [ 40.7245, -74.0044,  40.7308, -73.9927,   1.0000,   1.2089],
        [ 40.7719, -73.9554,  40.7630, -73.9676,   3.0000,   1.4274]])

In [29]:
y = torch.tensor(df[y_col].values, dtype=torch.float)

In [30]:
cats.shape

torch.Size([120000, 3])

In [31]:
conts.shape

torch.Size([120000, 6])

In [32]:
y.shape

torch.Size([120000, 1])

In [33]:
cat_szs = [len(df[col].cat.categories) for col in cat_cols]
cat_szs

[24, 2, 7]

In [35]:
emb_szs = [(size, min(50, (size+1)//2)) for size in cat_szs]
emb_szs

[(24, 12), (2, 1), (7, 4)]

In [36]:
catz = cats[:4]

In [37]:
catz

tensor([[ 4,  0,  1],
        [11,  0,  2],
        [ 7,  0,  2],
        [17,  1,  3]])

In [38]:
selfembeds = nn.ModuleList([nn.Embedding(ni,nf) for ni,nf in emb_szs])
selfembeds

ModuleList(
  (0): Embedding(24, 12)
  (1): Embedding(2, 1)
  (2): Embedding(7, 4)
)

In [ ]:
# Forward method
embeddingz = []

for i,e in enumerate(selfembeds):
    embeddingz.append(e(catz[:,i]))

embeddingz

[tensor([[ 0.8051,  0.4207,  0.2934,  0.6445,  1.1074,  0.0047, -0.8648, -0.3942,
          -0.6185,  0.2714, -0.2790, -1.5218],
         [-1.5402,  0.0861, -0.1825,  0.5026,  1.1681, -0.2863, -0.1214, -0.7738,
          -0.6690, -0.0599, -1.0187, -1.0986],
         [-0.4768, -0.1465, -2.8355,  0.7913,  1.0158,  1.4636,  0.4306,  0.6546,
          -0.7043,  1.2530,  0.4500,  0.4930],
         [ 0.2545, -1.2901, -0.2157,  0.1680,  1.9811, -0.0910, -0.8172, -0.3571,
           0.8305, -0.7479, -0.4807, -0.9718]], grad_fn=<EmbeddingBackward0>),
 tensor([[-0.4213],
         [-0.4213],
         [-0.4213],
         [-0.6878]], grad_fn=<EmbeddingBackward0>),
 tensor([[-1.2775,  0.6127,  0.4751, -0.2780],
         [-0.7986, -1.0617,  0.9345,  0.5501],
         [-0.7986, -1.0617,  0.9345,  0.5501],
         [-1.7906,  1.2721,  1.1367, -0.7497]], grad_fn=<EmbeddingBackward0>)]

In [41]:
z = torch.cat(embeddingz, 1)
z



tensor([[ 0.8051,  0.4207,  0.2934,  0.6445,  1.1074,  0.0047, -0.8648, -0.3942,
         -0.6185,  0.2714, -0.2790, -1.5218, -0.4213, -1.2775,  0.6127,  0.4751,
         -0.2780],
        [-1.5402,  0.0861, -0.1825,  0.5026,  1.1681, -0.2863, -0.1214, -0.7738,
         -0.6690, -0.0599, -1.0187, -1.0986, -0.4213, -0.7986, -1.0617,  0.9345,
          0.5501],
        [-0.4768, -0.1465, -2.8355,  0.7913,  1.0158,  1.4636,  0.4306,  0.6546,
         -0.7043,  1.2530,  0.4500,  0.4930, -0.4213, -0.7986, -1.0617,  0.9345,
          0.5501],
        [ 0.2545, -1.2901, -0.2157,  0.1680,  1.9811, -0.0910, -0.8172, -0.3571,
          0.8305, -0.7479, -0.4807, -0.9718, -0.6878, -1.7906,  1.2721,  1.1367,
         -0.7497]], grad_fn=<CatBackward0>)

In [42]:
selfembeddrop = nn.Dropout(0.4)

In [43]:
z = selfembeddrop(z)

In [44]:
z

tensor([[ 1.3418,  0.7012,  0.4890,  1.0742,  1.8457,  0.0079, -0.0000, -0.6570,
         -1.0308,  0.4524, -0.0000, -2.5363, -0.0000, -2.1291,  1.0211,  0.7918,
         -0.0000],
        [-2.5670,  0.1434, -0.3042,  0.8377,  1.9468, -0.0000, -0.2024, -1.2896,
         -0.0000, -0.0999, -0.0000, -0.0000, -0.7021, -1.3309, -0.0000,  0.0000,
          0.9168],
        [-0.7947, -0.2442, -0.0000,  0.0000,  1.6931,  2.4393,  0.0000,  0.0000,
         -0.0000,  0.0000,  0.0000,  0.8217, -0.7021, -1.3309, -0.0000,  0.0000,
          0.0000],
        [ 0.4241, -0.0000, -0.3595,  0.2801,  3.3019, -0.1517, -1.3620, -0.5952,
          1.3842, -0.0000, -0.8012, -1.6196, -0.0000, -0.0000,  2.1202,  0.0000,
         -1.2494]], grad_fn=<MulBackward0>)